In [11]:
!pip install -q transformers peft accelerate google-generativeai

In [12]:
import json
import time
import pandas as pd
from tqdm import tqdm
import google.generativeai as genai
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

In [ ]:
# Cấu hình API Gemini
GEMINI_API_KEY = ""  # Nhập API key của bạn
genai.configure(api_key="")

# Đường dẫn model và dữ liệu
MODEL_PATH = "/kaggle/input/qwen-finetuned/transformers/default/3/qwen_finetuned"
BASE_MODEL = "Qwen/Qwen3-4B"
QUESTIONS_FILE = "/kaggle/input/evaluate-quest/500_evaluation_questions.jsonl"  # File jsonl 602 câu hỏi
RESULTS_FILE = "/kaggle/working/evaluation_results.jsonl"
SUMMARY_FILE = "/kaggle/working/evaluation_summary.txt"

# Cấu hình generation
MAX_NEW_TOKENS = 1024
TEMPERATURE = 0.7
TOP_P = 0.9

In [14]:
print("Dang tai model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)
model = PeftModel.from_pretrained(base_model, MODEL_PATH)
print("Tai model thanh cong")

Dang tai model...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Tai model thanh cong


In [15]:
def load_questions(file_path):
    questions = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line.strip())
            questions.append(data)
    return questions

print("Dang tai cau hoi...")
questions = load_questions(QUESTIONS_FILE)
print(f"Da tai {len(questions)} cau hoi")

# Thong ke loai cau hoi
type_count = {}
for q in questions:
    q_type = q.get('type', 'unknown')
    type_count[q_type] = type_count.get(q_type, 0) + 1

print("Phan bo cau hoi:")
for q_type, count in type_count.items():
    print(f"  {q_type}: {count} cau")

Dang tai cau hoi...
Da tai 602 cau hoi
Phan bo cau hoi:
  insufficient_info: 120 cau
  off_topic: 78 cau
  out_of_domain: 164 cau
  in_domain: 240 cau


In [16]:
def generate_response(question_text, system_prompt="Ban la chuyen gia lich su Viet Nam."):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question_text}
    ]
    
    prompt = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs.input_ids.shape[-1]
    
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    new_tokens = outputs[0, input_len:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response.strip()

In [ ]:
# Cell: Sửa hàm đánh giá để ép JSON format
def evaluate_with_gemini(question, response, question_type):
    # Prompt ép buộc trả về JSON
    prompt = f"""
BẮT BUỘC: Chỉ trả về JSON, không có bất kỳ văn bản nào khác.

Đây là AI chỉ chuyên trả lời câu hỏi về lịch sử Việt Nam, sẽ có 4 loại câu hỏi chính cho AI bao gồm: IN-DOMAIN (trong chuyên môn), OUT-OF-DOMAIN (ngoài chuyên môn),INSUFFICIENT_INFO (thiếu thông tin), OFF-TOPIC (lạc đề). nhiệm vụ của bạn là Đánh giá câu trả lời AI lịch sử Việt Nam này theo các tiêu chí 0-1 điểm:

CÂU HỎI: {question}
LOẠI: {question_type}
CÂU TRẢ LỜI: {response}

Phân tích và trả về JSON với 4 tiêu chí:
- relevance: độ phù hợp
- accuracy: độ chính xác  
- helpfulness: tính hữu ích
- professionalism: tính chuyên nghiệp

VÍ DỤ OUTPUT:
{{"relevance": 0.8, "accuracy": 0.7, "helpfulness": 0.9, "professionalism": 0.8}}

BẮT BUỘC: Chỉ trả về JSON, không giải thích.
"""

    try:
        model = genai.GenerativeModel('gemini-2.0-flash')
        result = model.generate_content(prompt)
        result_text = result.text.strip()
        
        print(f"  [DEBUG] Raw Gemini response: {result_text}")
        
        # Làm sạch response - loại bỏ mọi text không phải JSON
        lines = result_text.split('\n')
        json_lines = []
        in_json = False
        
        for line in lines:
            line = line.strip()
            if line.startswith('{') or in_json:
                json_lines.append(line)
                in_json = True
            if line.endswith('}'):
                break
        
        if json_lines:
            json_str = ' '.join(json_lines)
            # Tìm vị trí { và } cuối cùng
            start_idx = json_str.find('{')
            end_idx = json_str.rfind('}') + 1
            
            if start_idx != -1 and end_idx != 0:
                json_str_clean = json_str[start_idx:end_idx]
                scores = json.loads(json_str_clean)
                print(f"  [DEBUG] Successfully parsed: {scores}")
                return scores
        
        # Fallback: cố gắng parse toàn bộ text
        try:
            scores = json.loads(result_text)
            print(f"  [DEBUG] Direct parse success: {scores}")
            return scores
        except:
            raise ValueError("Khong the parse JSON")
            
    except Exception as e:
        print(f"Loi danh gia: {e}")
        return {"relevance": 0.5, "accuracy": 0.5, "helpfulness": 0.5, "professionalism": 0.5}

print("Da cap nhat prompt ep JSON")

In [21]:
def calculate_overall_score(scores):
    weights = {
        "relevance": 0.3,
        "accuracy": 0.3,
        "helpfulness": 0.2,
        "professionalism": 0.2
    }
    
    total_score = 0
    total_weight = 0
    
    for criterion, weight in weights.items():
        if criterion in scores:
            total_score += scores[criterion] * weight
            total_weight += weight
    
    return total_score / total_weight if total_weight > 0 else 0.5

In [ ]:
print("Bat dau danh gia...")
print(f"Tong so cau hoi: {len(questions)}")

test_questions = questions[:100]  # Danh gia 100 cau dau tien
results = []
total_start_time = time.time()

for i, question_data in enumerate(test_questions):
    question_text = question_data['question']
    question_type = question_data.get('type', 'unknown')
    
    print(f"\n--- Cau hoi {i+1}/{len(test_questions)} ---")
    print(f"Loai: {question_type}")
    print(f"Noi dung: {question_text[:100]}...")
    
    # Generate response
    print("Dang generate response tu Qwen...")
    gen_start = time.time()
    try:
        response = generate_response(question_text)
        gen_time = time.time() - gen_start
        print(f"Generate hoan thanh trong {gen_time:.1f} giay")
        print(f"Response length: {len(response)} ky tu")
        print(f"Response preview: {response[:200]}...")
    except Exception as e:
        print(f"LOI khi generate: {e}")
        gen_time = time.time() - gen_start
        print(f"Generate that bai sau {gen_time:.1f} giay")
        response = "Loi generate response"
    
    # Danh gia voi Gemini
    print("Dang danh gia voi Gemini...")
    eval_start = time.time()
    try:
        evaluation_scores = evaluate_with_gemini(question_text, response, question_type)
        eval_time = time.time() - eval_start
        print(f"Danh gia hoan thanh trong {eval_time:.1f} giay")
        print(f"Diem danh gia: {evaluation_scores}")
        
        overall_score = calculate_overall_score(evaluation_scores)
        print(f"Diem tong: {overall_score:.2f}")
    except Exception as e:
        print(f"LOI khi danh gia: {e}")
        eval_time = time.time() - eval_start
        print(f"Danh gia that bai sau {eval_time:.1f} giay")
        evaluation_scores = {"error": str(e)}
        overall_score = 0.5
    
    # Luu ket qua
    result = {
        "question": question_text,
        "type": question_type,
        "response": response,
        "evaluation_scores": evaluation_scores,
        "overall_score": overall_score
    }
    
    results.append(result)
    
    # Tinh toan thoi gian trung binh va uoc tinh thoi gian con lai
    elapsed_time = time.time() - total_start_time
    avg_time_per_question = elapsed_time / (i + 1)
    remaining_questions = len(test_questions) - (i + 1)
    estimated_remaining_time = avg_time_per_question * remaining_questions
    
    print(f"Tien do: {i+1}/{len(test_questions)}")
    print(f"Thoi gian trung binh/cau: {avg_time_per_question:.1f}s")
    print(f"Uoc tinh con lai: {estimated_remaining_time/60:.1f} phut")
    
    # Luu ket qua sau moi 5 cau
    if (i + 1) % 5 == 0:
        with open(RESULTS_FILE, 'w', encoding='utf-8') as f:
            for r in results:
                f.write(json.dumps(r, ensure_ascii=False) + '\n')
        print(f"✓ Da luu ket qua tam thoi ({i + 1} cau)")
    
    # Nghi giua cac request
    if i < len(test_questions) - 1:  # Khong nghi sau cau cuoi cung
        sleep_time = 2
        print(f"Nghi {sleep_time} giay...")
        time.sleep(sleep_time)

total_time = time.time() - total_start_time
print(f"\n=== HOAN THANH DANH GIA ===")
print(f"Tong thoi gian: {total_time/60:.1f} phut")
print(f"Thoi gian trung binh/cau: {total_time/len(test_questions):.1f} giay")
print(f"Tong so cau da danh gia: {len(results)}")
print(f"File ket qua: {RESULTS_FILE}")

Bat dau danh gia...
Tong so cau hoi: 602

--- Cau hoi 1/100 ---
Loai: insufficient_info
Noi dung: Chính sách gì?...
Dang generate response tu Qwen...
Generate hoan thanh trong 13.4 giay
Response length: 504 ky tu
Response preview: <think>

</think>

Chào bạn, tôi rất sẵn lòng hỗ trợ bạn tìm hiểu về các chính sách liên quan đến lịch sử Việt Nam. Tuy nhiên, để tôi có thể cung cấp thông tin chính xác nhất, bạn có thể cho tôi biết ...
Dang danh gia voi Gemini...
Danh gia hoan thanh trong 0.7 giay
Diem danh gia: {'relevance': 0.8, 'accuracy': 0.7, 'helpfulness': 0.9, 'professionalism': 0.8}
Diem tong: 0.79
Tien do: 1/100
Thoi gian trung binh/cau: 14.1s
Uoc tinh con lai: 23.3 phut
Nghi 2 giay...

--- Cau hoi 2/100 ---
Loai: insufficient_info
Noi dung: Ai là người chỉ huy?...
Dang generate response tu Qwen...
Generate hoan thanh trong 96.2 giay
Response length: 3606 ky tu
Response preview: <think>

</think>

Câu hỏi của bạn về 'người chỉ huy' khá chung chung. Bạn có thể làm rõ hơn bạn đang mu

In [ ]:
# Luu ket qua cuoi cung
with open(RESULTS_FILE, 'w', encoding='utf-8') as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

print(f"Da luu ket qua vao: {RESULTS_FILE}")

In [ ]:
def create_summary_report(results):
    total_questions = len(results)
    avg_score = sum(r['overall_score'] for r in results) / total_questions
    
    # Phan bo theo loai
    type_scores = {}
    for r in results:
        q_type = r['type']
        if q_type not in type_scores:
            type_scores[q_type] = []
        type_scores[q_type].append(r['overall_score'])
    
    # Ghi bao cao
    with open(SUMMARY_FILE, 'w', encoding='utf-8') as f:
        f.write("BAO CAO DANH GIA MODEL\n")
        f.write("=" * 50 + "\n\n")
        f.write(f"Tong so cau hoi: {total_questions}\n")
        f.write(f"Diem trung binh: {avg_score:.2f}/1.0\n\n")
        
        f.write("DIEM THEO LOAI CAU HOI:\n")
        for q_type, scores in type_scores.items():
            avg = sum(scores) / len(scores)
            f.write(f"- {q_type}: {avg:.2f} ({len(scores)} cau)\n")
        
        # Top 5 cau hoi diem cao nhat
        f.write("\nTOP 5 CAU HOI DIEM CAO NHAT:\n")
        top_results = sorted(results, key=lambda x: x['overall_score'], reverse=True)[:5]
        for i, r in enumerate(top_results, 1):
            f.write(f"{i}. {r['question'][:100]}...\n")
            f.write(f"   Diem: {r['overall_score']:.2f}\n")

create_summary_report(results)
print(f"Da tao bao cao: {SUMMARY_FILE}")

In [ ]:
# Doc va hien thi ket qua
with open(RESULTS_FILE, 'r', encoding='utf-8') as f:
    results_data = [json.loads(line) for line in f]

print("KET QUA DANH GIA")
print("=" * 50)

avg_score = sum(r['overall_score'] for r in results_data) / len(results_data)
print(f"Diem trung binh: {avg_score:.2f}/1.0")
print(f"Tong so cau hoi da danh gia: {len(results_data)}")

print("\nPhan bo diem theo loai cau hoi:")
type_stats = {}
for r in results_data:
    q_type = r['type']
    if q_type not in type_stats:
        type_stats[q_type] = []
    type_stats[q_type].append(r['overall_score'])

for q_type, scores in type_stats.items():
    avg = sum(scores) / len(scores)
    print(f"  {q_type}: {avg:.2f} ({len(scores)} cau)")

print(f"\nKet qua chi tiet da luu vao: {RESULTS_FILE}")
print(f"Bao cao tong hop: {SUMMARY_FILE}")